![A soccer pitch for an international match.](soccer-pitch.jpg)

# Tests d'hypothèses avec des matchs de football masculin et féminin (version R)

Vous travaillez comme journaliste sportif dans un grand média en ligne, spécialisé dans l'analyse et le reportage sur le football. Vous suivez depuis plusieurs années des matches internationaux féminins et masculins, et votre intuition vous dit qu'il se marque plus de buts dans les rencontres internationales féminines que masculines. Le sujet ferait un excellent article d'enquête que vos abonnés vont adorer, mais il vous faut mener un test d'hypothèse statistique valide pour en avoir le cœur net !

En cadrant ce projet, vous notez que le football a beaucoup évolué au fil des ans, et que les performances varient sans doute fortement selon les compétitions. Vous décidez donc de limiter les données utilisées dans l'analyse aux seuls matches officiels de `FIFA World Cup` (hors qualifications) disputés après le `2002-01-01`.

Vous avez constitué deux jeux de données rassemblant les résultats de tous les matches internationaux féminins et masculins officiels depuis le XIXᵉ siècle, extraits d'une source en ligne fiable. Ces données sont stockées dans deux fichiers CSV : `women_results.csv` et `men_results.csv`.

La question à laquelle vous cherchez à répondre est :

> Marque-t-on plus de buts dans les matches internationaux de football féminin que masculin ?

Vous retenez un **seuil de signification de 10 %**, et utilisez les hypothèses nulle et alternative suivantes :

$H_0$ : le nombre moyen de buts marqués lors des matches internationaux féminins est identique à celui des matches masculins.

$H_A$ : le nombre moyen de buts marqués lors des matches internationaux féminins est supérieur à celui des matches masculins.


In [ ]:
# Import necessary libraries
library(tidyverse)


In [1]:
# Start your code here!
# Use as many cells as you like

# Charger les données
women_results <- read_csv("women_results.csv", show_col_types = FALSE)
men_results <- read_csv("men_results.csv", show_col_types = FALSE)

# Filtrer : FIFA World Cup uniquement, depuis le 2002-01-01
women_subset <- women_results %>%
  filter(date > "2002-01-01", tournament == "FIFA World Cup") %>%
  mutate(total_goals = home_score + away_score)

men_subset <- men_results %>%
  filter(date > "2002-01-01", tournament == "FIFA World Cup") %>%
  mutate(total_goals = home_score + away_score)

# Test de Wilcoxon (Mann-Whitney U) : H_A -> femmes marquent plus que hommes
test_result <- wilcox.test(
  x = women_subset$total_goals,
  y = men_subset$total_goals,
  alternative = "greater"
)

# Extraire la p-value
p_val <- test_result$p.value

# Décision au seuil de 10 %
result <- ifelse(p_val <= 0.10, "reject", "fail to reject")

# Stocker le résultat dans un data frame
result_df <- data.frame(p_val = p_val, result = result)

print(result_df)


  p_val result
1 0.00510661 reject


## Interprétation des résultats

Le test de Wilcoxon / Mann-Whitney U (choisi car le nombre de buts par match suit une distribution asymétrique et non normale, ce qui exclut un test t classique) renvoie une **p-value d'environ 0.0051 (0.51 %)**.

Comme cette p-value est largement inférieure au seuil de signification retenu de **10 % (0.10)**, nous **rejetons l'hypothèse nulle H₀** en faveur de l'hypothèse alternative H_A.

**Conclusion :** au regard des matchs officiels de la FIFA World Cup joués depuis le 2002-01-01, le nombre moyen de buts marqués lors des matchs internationaux **féminins** est **statistiquement significativement supérieur** à celui des matchs **masculins**. Ce résultat est cohérent avec celui obtenu via l'analyse Python équivalente (test de Mann-Whitney U avec `pingouin`), ce qui confirme la robustesse de la conclusion indépendamment du langage ou de la librairie utilisée.

À noter : la p-value obtenue est si faible qu'elle resterait significative même à un seuil beaucoup plus strict (1 %), ce qui renforce la solidité de cette conclusion.
